[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tymworld/DROPPS/blob/v1.0.0/examples/colab/03_DROPPS_1_0_Trajectory_Preparation.ipynb)

# DROPPS 1.0 Trajectory Preparation and Indexing

Prepare the **elongated-box NVT trajectory** for analysis. The notebook rejects
an NPT run input, creates reusable NDX groups through the public `make_ndx`
interface, and applies PBC reconstruction, centering, fitting, time selection,
XTC/PDB conversion, and snapshot extraction with `trjconv`.

Transform order in DROPPS 1.0 is: make whole/no-jump → center → pack → fit →
translate. Time selections use physical units and choose the nearest saved frame.


In [ ]:
#@title Install DROPPS 1.0
#@markdown The default route installs the immutable wheel attached to the
#@markdown public GitHub `v1.0.0` release. Upload remains available for offline use.
installation_source = "Install from the DROPPS v1.0.0 GitHub release" #@param ["Install from the DROPPS v1.0.0 GitHub release", "Upload the DROPPS 1.0 wheel", "Install from another wheel URL"]
wheel_url = "https://github.com/tymworld/DROPPS/releases/download/v1.0.0/dropps-1.0-py3-none-any.whl" #@param {type:"string"}
accelerator_dependencies = "auto" #@param ["auto", "cuda12", "cuda13", "latest"]

import importlib.metadata
import re
import shutil
import subprocess
import sys
from pathlib import Path


def detect_install_extra():
    if accelerator_dependencies != "auto":
        return accelerator_dependencies
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(
            ["nvidia-smi"], text=True, capture_output=True, check=False
        ).stdout
        match = re.search(r"CUDA Version:\s*(\d+)", probe)
        if match and int(match.group(1)) >= 13:
            return "cuda13"
        return "cuda12"
    return "latest"


if installation_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "This upload form is intended for Google Colab. Set installation_source "
            "to the URL option when running elsewhere."
        ) from exc
    uploaded = files.upload()
    wheel_candidates = [Path(name) for name in uploaded if name.endswith(".whl")]
    if len(wheel_candidates) != 1:
        raise ValueError("Upload exactly one DROPPS 1.0 .whl file.")
    wheel_target = str(wheel_candidates[0].resolve())
else:
    if not wheel_url.strip():
        raise ValueError("Provide the published DROPPS 1.0 wheel URL.")
    wheel_target = wheel_url.strip()

extra = detect_install_extra()
if wheel_target.startswith(("https://", "http://")):
    package_spec = f"dropps[{extra}] @ {wheel_target}"
else:
    package_spec = f"{wheel_target}[{extra}]"
print(f"Installing {package_spec}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        package_spec,
    ],
    check=True,
)

version = importlib.metadata.version("dropps")
if version != "1.0":
    raise RuntimeError(f"Expected DROPPS 1.0, but installed {version}.")
subprocess.run(["dps", "--version"], check=True)


In [ ]:
#@title Shared notebook helpers
import hashlib
import json
import os
import shlex
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

COMMAND_LOG = []


def run_command(arguments, *, cwd=None, input_text=None, check=True):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    COMMAND_LOG.append(
        {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "cwd": str(Path(cwd or Path.cwd()).resolve()),
            "command": command,
            "returncode": result.returncode,
        }
    )
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        )
    return result


def dps(*arguments, cwd=None, input_text=None, check=True):
    return run_command(
        ["dps", *arguments], cwd=cwd, input_text=input_text, check=check
    )


def reset_task_directory(path):
    path = Path(path).resolve()
    if not path.name.startswith("dropps_"):
        raise ValueError(f"Refusing to reset unexpected directory: {path}")
    if path.exists():
        import shutil

        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        bundle.extractall(destination)


def find_unique(root, basename):
    matches = [path for path in Path(root).rglob(basename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {basename!r} below {root}, found {len(matches)}."
        )
    return matches[0]


def make_zip(paths, output_path, *, base=None):
    output_path = Path(output_path)
    base = Path(base or output_path.parent).resolve()
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted({Path(path).resolve() for path in paths}):
            if source == output_path.resolve() or not source.is_file():
                continue
            try:
                arcname = source.relative_to(base)
            except ValueError:
                arcname = Path(source.name)
            bundle.write(source, arcname)
    print(f"Wrote {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
    return output_path


def download(path):
    try:
        from google.colab import files
    except ImportError:
        print(f"Result available at {Path(path).resolve()}")
    else:
        files.download(str(path))


def optional_time_arguments(start, end, interval):
    arguments = []
    for option, value in (("-b", start), ("-e", end), ("-dt", interval)):
        if str(value).strip():
            arguments.extend([option, str(value).strip()])
    return arguments


def read_xvg(path):
    legends = {}
    metadata = {}
    rows = []
    with Path(path).open(encoding="utf-8") as stream:
        for raw in stream:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("@"):
                re_module = __import__("re")
                match = re_module.search(r's(\d+)\s+legend\s+"(.*)"', line)
                if match:
                    legends[int(match.group(1)) + 1] = match.group(2)
                for key, pattern in {
                    "title": r'@\s+title\s+"(.*)"',
                    "xlabel": r'@\s+xaxis\s+label\s+"(.*)"',
                    "ylabel": r'@\s+yaxis\s+label\s+"(.*)"',
                }.items():
                    label_match = re_module.search(pattern, line)
                    if label_match:
                        metadata[key] = label_match.group(1)
                continue
            rows.append([float(value) for value in line.split()])
    data = np.asarray(rows, dtype=float)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"No plottable data in {path}")
    return data, legends, metadata


def plot_xvg(path, *, title=None, output=None):
    data, legends, metadata = read_xvg(path)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for column in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, column], label=legends.get(column, f"series {column}"))
    ax.set_title(title or metadata.get("title", Path(path).stem))
    ax.set_xlabel(metadata.get("xlabel", "x / time"))
    ax.set_ylabel(metadata.get("ylabel", "value"))
    if data.shape[1] > 2:
        ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    output_path = Path(output) if output is not None else Path(path).with_suffix(".png")
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    plt.show()
    return fig, output_path


In [ ]:
#@title Upload or reuse a workflow bundle
input_source = "Upload ZIP or individual files" #@param ["Upload ZIP or individual files", "Reuse files from an earlier notebook in this runtime"]
reuse_directory = "/content/dropps_simulation" #@param {type:"string"}

import shutil

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
INPUT_ROOT = reset_task_directory(BASE / "dropps_uploaded_inputs")

if input_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload this bundle in Colab or choose the reuse option.") from exc
    uploaded = files.upload()
    for name in uploaded:
        source = Path(name)
        if source.suffix.lower() == ".zip":
            safe_extract_zip(source, INPUT_ROOT)
        else:
            shutil.copy2(source, INPUT_ROOT / source.name)
else:
    source_root = Path(reuse_directory)
    if not source_root.is_dir():
        raise FileNotFoundError(source_root)
    for source in source_root.rglob("*"):
        if source.is_file() and source.suffix.lower() in {
            ".pdb", ".itp", ".top", ".mdp", ".tpr", ".xtc", ".ndx", ".chk", ".xml", ".json"
        }:
            target = INPUT_ROOT / source.name
            if target.exists() and sha256(target) != sha256(source):
                raise ValueError(f"Duplicate filename with different content: {source.name}")
            shutil.copy2(source, target)

print("Available files:")
for path in sorted(INPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(INPUT_ROOT), f"({path.stat().st_size / 1024:.1f} KiB)")


## 1 Resolve and validate the run files


In [ ]:
#@title Run filenames
run_input_filename = "slab_nvt.tpr" #@param {type:"string"}
trajectory_filename = "slab_nvt.xtc" #@param {type:"string"}

TPR = find_unique(INPUT_ROOT, run_input_filename)
TRAJECTORY = find_unique(INPUT_ROOT, trajectory_filename)

import zipfile

if not zipfile.is_zipfile(TPR):
    raise ValueError("Expected a portable DROPPS 1.0 TPR v2 file.")
with zipfile.ZipFile(TPR) as archive:
    tpr_parameters = json.loads(archive.read("parameters.json"))
if tpr_parameters.get("pcoulp"):
    raise ValueError(
        "Trajectory preparation requires the elongated-box NVT TPR, not the NPT TPR."
    )

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORKDIR = reset_task_directory(BASE / "dropps_trajectory")
import shutil
ANALYSIS_TPR = WORKDIR / "slab_nvt.tpr"
shutil.copy2(TPR, ANALYSIS_TPR)
dps("check", "-s", TPR, "-f", TRAJECTORY, cwd=WORKDIR)
print("Verified ensemble: NVT (pcoulp = False)")


## 2 Create an index file

Commands are separated by semicolons. Initial groups are `System` (group 0)
followed by one group per molecule type. New selections append groups in order.
Example: `mol SCAFFOLD; name 3 Scaffold_all; resid 1-4 & mol SCAFFOLD;
name 4 Scaffold_N; q`. The notebook adds `q` if omitted.


In [ ]:
#@title Non-interactive make_ndx command script
index_commands = "q" #@param {type:"string"}
index_filename = "analysis.ndx" #@param {type:"string"}

commands = [part.strip() for part in index_commands.split(";") if part.strip()]
if not commands or commands[-1] != "q":
    commands.append("q")
NDX = WORKDIR / index_filename
result = dps(
    "make_ndx", "-s", TPR, "-o", NDX,
    cwd=WORKDIR, input_text="\n".join(commands) + "\n",
    check=False,
)
if not NDX.is_file():
    raise RuntimeError(
        f"make_ndx did not create {NDX}; subprocess status was {result.returncode}."
    )
if result.returncode:
    print(
        "Note: DROPPS 1.0 make_ndx writes the requested file and then exits "
        "through its legacy interactive quit path, which reports status 1."
    )
dps("check", "-s", TPR, "-f", TRAJECTORY, "-n", NDX, cwd=WORKDIR)
print(NDX.read_text(encoding="utf-8")[:4000])


## 3 Convert and transform the trajectory


In [ ]:
#@title trjconv settings
output_filename = "processed.xtc" #@param {type:"string"}
output_selection = "group 0" #@param {type:"string"}
start_time = "" #@param {type:"string"}
end_time = "" #@param {type:"string"}
sampling_interval = "" #@param {type:"string"}
time_unit = "ns" #@param ["fs", "ps", "ns", "us", "ms", "s"]
pbc_mode = "whole" #@param ["none", "whole", "atom", "res", "mol", "nojump"]
center_mode = "none" #@param ["none", "geometry", "mass", "dense"]
center_selection = "" #@param {type:"string"}
center_axis = "xyz" #@param ["x", "y", "z", "xyz"]
dense_phase_threshold = 0.5 #@param {type:"number"}
density_bin_width_nm = 0.05 #@param {type:"number"}
fit_mode = "none" #@param ["none", "translation", "transxy", "rot+trans", "rotxy+transxy", "progressive"]
fit_selection = "" #@param {type:"string"}
fit_reference = "tpr" #@param ["tpr", "first"]
fit_weighting = "mass" #@param ["mass", "uniform"]
translation_nm = "" #@param {type:"string"}
per_frame_shift_nm = "" #@param {type:"string"}
xtc_precision_decimals = 3 #@param {type:"integer"}

OUTPUT_TRAJECTORY = WORKDIR / output_filename
arguments = [
    "trjconv", "-s", TPR, "-f", TRAJECTORY, "-n", NDX,
    "-sel", output_selection, "-o", OUTPUT_TRAJECTORY,
    "-tu", time_unit, "-pbc", pbc_mode, "--center", center_mode,
    "--center-axis", center_axis, "-dpt", dense_phase_threshold,
    "--density-bin-width", density_bin_width_nm,
    "-fit", fit_mode, "--fit-reference", fit_reference,
    "--fit-weighting", fit_weighting, "-ndec", xtc_precision_decimals,
    *optional_time_arguments(start_time, end_time, sampling_interval),
]
if center_selection.strip():
    arguments.extend(["--center-select", center_selection.strip()])
if fit_selection.strip():
    arguments.extend(["--fit-select", fit_selection.strip()])
if translation_nm.strip():
    vector = translation_nm.replace(",", " ").split()
    if len(vector) != 3:
        raise ValueError("translation_nm requires three values")
    arguments.extend(["-trans", *vector])
if per_frame_shift_nm.strip():
    vector = per_frame_shift_nm.replace(",", " ").split()
    if len(vector) != 3:
        raise ValueError("per_frame_shift_nm requires three values")
    arguments.extend(["-shift", *vector])
dps(*arguments, cwd=WORKDIR)
dps("check", "-s", TPR, "-f", OUTPUT_TRAJECTORY, "-n", NDX, cwd=WORKDIR)


In [ ]:
#@title Extract a PDB snapshot
extract_snapshot = True #@param {type:"boolean"}
snapshot_time = 0.0 #@param {type:"number"}
snapshot_time_unit = "ns" #@param ["fs", "ps", "ns", "us", "ms", "s"]
snapshot_selection = "group 0" #@param {type:"string"}
add_conect_records = True #@param {type:"boolean"}

if extract_snapshot:
    SNAPSHOT = WORKDIR / "snapshot.pdb"
    arguments = [
        "trjconv", "-s", TPR, "-f", OUTPUT_TRAJECTORY, "-n", NDX,
        "-sel", snapshot_selection, "-o", SNAPSHOT,
        "-b", snapshot_time, "-e", snapshot_time, "-tu", snapshot_time_unit,
        "-pbc", "whole",
    ]
    if add_conect_records:
        arguments.append("--conect")
    dps(*arguments, cwd=WORKDIR)
    print("Snapshot:", SNAPSHOT)


## 4 Export the prepared trajectory bundle


In [ ]:
#@title Save manifest and download
manifest = {
    "notebook": "03_DROPPS_1_0_Trajectory_Preparation.ipynb",
    "dropps_version": "1.0",
    "source_tpr_sha256": sha256(TPR),
    "source_trajectory_sha256": sha256(TRAJECTORY),
    "verified_ensemble": "NVT",
    "index_commands": commands,
    "commands": COMMAND_LOG,
}
manifest_path = WORKDIR / "trajectory_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
export_files = [ANALYSIS_TPR, NDX, OUTPUT_TRAJECTORY, manifest_path]
if extract_snapshot:
    export_files.append(SNAPSHOT)
archive = make_zip(
    export_files,
    WORKDIR / "DROPPS_1_0_trajectory_bundle.zip",
    base=WORKDIR,
)
download(archive)
